# Estimación de IMC desde imágenes faciales — Notebook consolidado

Este notebook reúne todo el proyecto en un solo lugar y es **autosuficiente** para
las partes bayesianas y de incertidumbre: las clases del modelo, el bucle de
entrenamiento ELBO, el Monte Carlo sampling y las tres métricas de confianza
están **inline** (no dependen de `src/`). Las partes deterministas grandes hacen
llamada al módulo `src/` empaquetado dentro de este mismo entregable.

**Secciones:**

| # | Sección | Implementación |
|---|---|---|
| 0 | Setup, paths, GPU | inline |
| 1 | Carga del dataset + exploración | inline |
| 2 | Modelo bayesiano (`BayesianBMIRegressor`) | **inline completo** |
| 3 | Entrenamiento bayesiano (ELBO) | **inline completo** |
| 4 | Monte Carlo sampling — núcleo del análisis bayesiano | **inline completo** |
| 5 | Las 3 métricas de confianza (var, entropía, MI) | **inline completo** |
| 6 | Score compuesto + validación (risk-coverage curve) | **inline completo** |
| 7 | MC dropout sobre el clasificador determinista 3-clases | **inline completo** |
| 8 | Resultados sobre VisualBMI test e IDOC zero-shot | inline |
| 9 | Entrenamientos deterministas (ViT-L, etc.) | referencia a `src/` |
| 10 | Generación de los 9 plots PNG | referencia a `src/` |

Por defecto los flags `TRAIN_*` están en `False` → el notebook se ejecuta solo
analizando los checkpoints existentes y los caches MC en `cache/` (5 min).


## 0. Setup

Imports, configuración de paths y verificación de GPU.

In [ ]:
import sys, os, copy, math, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

# Detectar la raíz del proyecto (subir hasta encontrar "BMI proyecto")
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "BMI proyecto" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

# src/ se incluye como respaldo (los entrenamientos deterministas grandes lo usan)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "entregable"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("torch       :", torch.__version__)
print("device      :", device,
      "({})".format(torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"))


### Parámetros globales

Una sola celda con todos los hiperparámetros que se pueden tocar. Cambiar aquí en
lugar de buscar números mágicos repartidos por el notebook.

In [ ]:
# >>> PARÁMETROS EDITABLES <<<
CSV_PATH       = PROJECT_ROOT / "visual_bmi_dedup.csv"   # VisualBMI deduplicado (sin Zheng, sin leakage)
IMAGE_SIZE     = 224
BATCH_SIZE     = 32
NUM_WORKERS    = 4
SEEDS          = [42, 123, 456, 789]                     # 4 seeds para el ensemble
RUNS_DIR       = PROJECT_ROOT / "runs"                   # checkpoints existentes
CACHE_DIR      = PROJECT_ROOT / "entregable" / "cache"

# Activar para reentrenar (no se necesita si ya tenemos los checkpoints en runs/)
TRAIN_BAYES_REG  = False     # entrena 1 regresor bayesiano (~80 min por seed)

# Parámetros del análisis de incertidumbre
N_MC_SAMPLES   = 30          # forwards Monte Carlo por modelo
KERNEL_WIDTH   = 2.0         # ancho del kernel gaussiano para marginalización OMS (kg/m²)

# Límites OMS estándar (kg/m²)
OMS_BOUNDS_4 = [-float("inf"), 25.0, 30.0, 35.0, float("inf")]
OMS_NAMES_4  = ["Normal", "Sobrepeso", "Obesidad I", "Obesidad II+"]
OMS_BOUNDS_3 = [-float("inf"), 25.0, 30.0, float("inf")]
OMS_NAMES_3  = ["Normal", "Sobrepeso", "Obesidad"]


## 1. Carga del dataset

El CSV combinado tiene columnas: `image_path, BMI, is_female, person_id, source,
is_training`. La partición train/test viene marcada por `is_training`. El val se
extrae del train **a nivel de persona** (no de imagen) para evitar data leakage
entre fotos de la misma persona.

In [ ]:
df_full = pd.read_csv(CSV_PATH)
print(f"Total: {len(df_full)} imágenes | {df_full['person_id'].nunique()} personas")
print(f"BMI: rango [{df_full['BMI'].min():.1f}, {df_full['BMI'].max():.1f}], media {df_full['BMI'].mean():.2f}")
print(f"is_training counts: {df_full['is_training'].value_counts().to_dict()}")
print(f"Female %: {df_full['is_female'].mean()*100:.1f}")

# Partición a nivel de persona — DETERMINISTA con seed 42
def person_split(df, val_ratio=0.10, seed=42):
    train_df = df[df["is_training"] == 1].reset_index(drop=True)
    test_df  = df[df["is_training"] == 0].reset_index(drop=True)
    persons = sorted(train_df["person_id"].drop_duplicates())
    rng = np.random.default_rng(seed)
    rng.shuffle(persons)
    n_val = max(1, int(round(val_ratio * len(persons))))
    val_ids = set(persons[:n_val])
    tr = train_df[~train_df["person_id"].isin(val_ids)].reset_index(drop=True)
    vl = train_df[train_df["person_id"].isin(val_ids)].reset_index(drop=True)
    # Asserts críticos: no fuga entre splits
    assert set(tr["person_id"]).isdisjoint(set(vl["person_id"])), "leak train↔val"
    assert set(tr["person_id"]).isdisjoint(set(test_df["person_id"])), "leak train↔test"
    return tr, vl, test_df

train_df, val_df, test_df = person_split(df_full, val_ratio=0.10, seed=42)
print(f"\nSplits: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


### Inspección rápida del dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, dsplit, name, color in zip(axes, [train_df, val_df, test_df],
                                    ["Train", "Val", "Test"],
                                    ["#27ae60", "#f39c12", "#e74c3c"]):
    ax.hist(dsplit["BMI"], bins=30, color=color, alpha=0.85, edgecolor="white")
    for b in [25, 30, 35]:
        ax.axvline(b, color="black", ls="--", lw=1, alpha=0.5)
    ax.set_title(f"{name} (n={len(dsplit)})", fontweight="bold")
    ax.set_xlabel("BMI"); ax.set_ylabel("# imgs")
plt.tight_layout(); plt.show()


## 2. Modelo bayesiano — definición inline

Aquí está el corazón del proyecto bayesiano. La idea: cada peso de la cabeza no es
un número fijo sino una **distribución normal** `N(μ, σ²)`. En cada forward pass
se sortea un set distinto de pesos (truco de reparametrización) → distintas
predicciones para la misma imagen. Las clases siguientes son la implementación
exacta de `src/model.py`.

In [ ]:
import timm
from bayesian_torch.layers import LinearReparameterization

class BayesianHead(nn.Module):
    """
    Cabeza MLP con dos capas variacionales.

    En cada forward:
      1. fc1 samplea pesos w_1 ~ N(μ_1, σ_1²) y devuelve (activación, kl_1)
      2. ReLU + Dropout
      3. fc2 samplea pesos w_2 ~ N(μ_2, σ_2²) y devuelve (output, kl_2)
      4. Retorna (output, kl_total)  donde kl_total = kl_1 + kl_2

    El término KL se suma a la loss del modelo para regularizar la posterior
    contra la prior N(0, 1).
    """
    def __init__(self, n_features: int, num_classes: int = 1, dropout: float = 0.3):
        super().__init__()
        self.num_classes = num_classes
        self.fc1  = LinearReparameterization(n_features, 256)
        self.act  = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2  = LinearReparameterization(256, num_classes)

    def forward(self, x):
        out1, kl1 = self.fc1(x)
        out2, kl2 = self.fc2(self.drop(self.act(out1)))
        kl = kl1 + kl2
        if self.num_classes == 1:
            return out2.squeeze(1), kl     # regresión: ([B], escalar)
        return out2, kl                    # clasificación: ([B,C], escalar)


class BayesianBMIRegressor(nn.Module):
    """
    Backbone timm determinístico (ViT-B) + cabeza variacional BayesianHead.

    Solo la cabeza es bayesiana: el backbone se queda determinístico para no
    inflar el costo computacional y porque los features pre-entrenados de
    ImageNet ya son buenos.
    """
    def __init__(self, backbone_name="vit_base_patch16_224.augreg_in21k_ft_in1k",
                 pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = BayesianHead(self.backbone.num_features, num_classes=1, dropout=dropout)

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)        # (preds [B], kl escalar)


def build_bayes_regressor(backbone="vit_base_patch16_224.augreg_in21k_ft_in1k",
                          pretrained=True, dropout=0.3):
    m = BayesianBMIRegressor(backbone_name=backbone, pretrained=pretrained, dropout=dropout)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"BayesianBMIRegressor: {backbone} ({n_params:.1f} M params)")
    return m


## 3. Entrenamiento bayesiano — loop ELBO inline

La función de pérdida cambia de `L1Loss` puro a **ELBO** (Evidence Lower BOund):

$$\mathcal{L} = \text{L1Loss}(\hat{y}, y) + \beta \cdot \text{KL}(q(w) \| p(w))$$

donde `β = 1 / N_batches` normaliza la divergencia KL contra la prior `N(0, 1)`.
La KL regulariza la posterior para no alejarse demasiado de la prior — actúa como
regularizador bayesiano que reemplaza al weight decay clásico.

**AMP debe estar desactivado** (`fp32` obligatorio) porque las varianzas σ² pueden
sufrir underflow en fp16. Si σ → 0, la capa variacional degenera a determinista.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class ModelEMA:
    """Exponential Moving Average de los pesos del modelo (decay=0.999)."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.shadow.parameters(), model.parameters()):
            ema_p.mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)
        for ema_b, b in zip(self.shadow.buffers(), model.buffers()):
            ema_b.copy_(b)


def run_epoch_bayes(model, loader, criterion, optimizer, device, kl_weight,
                    train=True, grad_clip=1.0, ema=None):
    """Una época con loss ELBO. Devuelve dict con mae, kl, total loss."""
    model.train(train)
    all_preds, all_targets, total_loss = [], [], 0.0
    with torch.set_grad_enabled(train):
        for images, bmis in loader:
            images = images.to(device, non_blocking=True)
            bmis   = bmis.to(device, non_blocking=True)
            preds, kl = model(images)
            task_loss = criterion(preds, bmis)
            loss = task_loss + kl_weight * kl
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if ema is not None:
                    ema.update(model)
            total_loss += loss.item() * len(bmis)
            all_preds   += preds.detach().cpu().float().tolist()
            all_targets += bmis.cpu().float().tolist()
    mae = float(np.abs(np.array(all_preds) - np.array(all_targets)).mean())
    return {"mae": mae, "loss": total_loss / len(all_targets)}


### Lanzar el entrenamiento bayesiano

Solo se ejecuta si `TRAIN_BAYES_REG=True` en la sección 0. Cada seed produce un
checkpoint en `runs/visualbmi_bayes_seed{seed}/best.pt`. Cada seed toma ~80 min en
RTX 5080 (fp32, sin AMP).

In [ ]:
if TRAIN_BAYES_REG:
    # Para no duplicar 600 líneas: importamos las utilidades de dataloaders desde src/
    # (estas son deterministas y compartidas con el regresor non-bayes)
    from src.dataset import VisualBMIDataset, get_transforms, make_gender_balanced_sampler
    from torch.utils.data import DataLoader

    EPOCHS, FREEZE_EPOCHS, PATIENCE = 60, 10, 10
    LR, DROPOUT = 1e-4, 0.3

    for seed in SEEDS:
        print(f"\n{'='*60}\n  Bayesian seed={seed}\n{'='*60}")
        set_seed(seed)

        # DataLoaders
        train_ds = VisualBMIDataset(df=train_df, transform=get_transforms("train", IMAGE_SIZE))
        val_ds   = VisualBMIDataset(df=val_df,   transform=get_transforms("val",   IMAGE_SIZE))
        sampler  = make_gender_balanced_sampler(train_df)
        train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)

        # Modelo y optimización
        model = build_bayes_regressor(dropout=DROPOUT).to(device)
        # Congelar backbone los primeros FREEZE_EPOCHS épocas
        for p in model.backbone.parameters():
            p.requires_grad = False
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        criterion = nn.L1Loss()
        ema = ModelEMA(model, decay=0.999)
        kl_weight = 1.0 / len(train_loader)

        out_dir = RUNS_DIR / f"visualbmi_bayes_seed{seed}"; out_dir.mkdir(parents=True, exist_ok=True)
        best_val_mae, patience_left = float("inf"), PATIENCE

        for epoch in range(1, EPOCHS + 1):
            if epoch == FREEZE_EPOCHS + 1:
                for p in model.backbone.parameters(): p.requires_grad = True
                optimizer = torch.optim.AdamW(model.parameters(), lr=LR/10, weight_decay=1e-4)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - FREEZE_EPOCHS)
                ema = ModelEMA(model, decay=0.999)
            t0 = time.time()
            tr = run_epoch_bayes(model, train_loader, criterion, optimizer, device,
                                  kl_weight, train=True, ema=ema)
            vl = run_epoch_bayes(ema.shadow, val_loader, criterion, optimizer, device,
                                  kl_weight, train=False)
            scheduler.step()
            print(f"Ep {epoch:03d}  train MAE={tr['mae']:.3f}  val MAE={vl['mae']:.3f}  "
                  f"loss={tr['loss']:.3f}  ({time.time()-t0:.0f}s)")
            if vl["mae"] < best_val_mae:
                best_val_mae = vl["mae"]; patience_left = PATIENCE
                torch.save({"model": ema.state_dict(), "epoch": epoch,
                            "val_mae": best_val_mae,
                            "config": {"backbone": "vit_base_patch16_224.augreg_in21k_ft_in1k",
                                       "dropout": DROPOUT}, "ema": True},
                           out_dir / "best.pt")
            else:
                patience_left -= 1
                if patience_left == 0:
                    print(f"  Early stopping at epoch {epoch}"); break
        print(f"\nBest val MAE: {best_val_mae:.4f}")
else:
    print("Saltado (TRAIN_BAYES_REG=False). Usaremos los checkpoints existentes.")


## 4. Monte Carlo sampling — núcleo del análisis bayesiano

> *"Correr el modelo muchas veces y ver cuántas veces dice que sí."*

Eso es **Monte Carlo sampling** sobre el modelo bayesiano. En el modelo cada peso
es una distribución; cada forward sortea pesos distintos. Con N=30 forwards por
imagen tenemos una "distribución empírica" de la predicción.

**Detalle crítico — `model.train()`**: para que las capas variacionales sigan
sorteando pesos en inferencia hay que ponerlas en modo `train()`, no `eval()`. En
`eval()` `LinearReparameterization` devuelve solo la media μ y deja de samplear.

Con 4 seeds de modelos bayesianos × 30 muestras MC = **120 muestras por imagen**.

In [ ]:
@torch.no_grad()
def mc_predict(model, dataloader, n_samples=30, device="cuda"):
    """Devuelve (mc_samples [n_samples, N_imgs], targets [N_imgs])."""
    model.train()   # activa el sampling estocástico (esto NO es un bug)
    all_samples, all_targets = [], []
    for images, bmis in dataloader:
        images = images.to(device, non_blocking=True)
        # n_samples forwards sobre el mismo batch
        samples = torch.stack([model(images)[0] for _ in range(n_samples)], dim=0)
        all_samples.append(samples.cpu().numpy())
        all_targets.append(bmis.numpy())
    return np.concatenate(all_samples, axis=1), np.concatenate(all_targets, axis=0)


def load_bayes_ensemble(seeds=SEEDS, runs_dir=RUNS_DIR, device=device):
    """Carga los 4 regresores bayesianos pre-entrenados."""
    models = []
    for seed in seeds:
        path = runs_dir / f"visualbmi_bayes_seed{seed}" / "best.pt"
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        cfg  = ckpt.get("config", {})
        m = build_bayes_regressor(
            backbone=cfg.get("backbone", "vit_base_patch16_224.augreg_in21k_ft_in1k"),
            pretrained=False, dropout=float(cfg.get("dropout", 0.3))
        ).to(device)
        m.load_state_dict(ckpt["model"]); m.train()
        models.append(m)
        print(f"  seed={seed}  val_mae={ckpt.get('val_mae','?'):.4f}")
    return models


def mc_predict_ensemble(models, dataloader, n_samples_per_model=30, device="cuda"):
    """Llama a mc_predict sobre cada modelo y concatena → [N_models*n_samples, N_imgs]."""
    all_mc, targets = [], None
    for i, m in enumerate(models):
        print(f"  model {i+1}/{len(models)} ({n_samples_per_model} MC passes)...")
        mc, tg = mc_predict(m, dataloader, n_samples_per_model, device)
        all_mc.append(mc)
        targets = tg if targets is None else targets
    return np.concatenate(all_mc, axis=0), targets


## 5. Las tres métricas de confianza (inline)

A partir de las `M = 120` predicciones por imagen calculamos:

**(a) Varianza predictiva** — dispersión de las 120 predicciones numéricas:
$$\sigma^2 = \frac{1}{M} \sum_i (\mu_i - \bar{\mu})^2$$

**(b) Entropía categórica** — cada $\mu_i$ se "suaviza" sobre clases OMS con un
kernel gaussiano de ancho $s = 2$ kg/m²:
$$p_i(c) = \int_{lo_c}^{hi_c} \mathcal{N}(y;\mu_i,s^2)\,dy$$
Luego promediamos las 120 distribuciones y calculamos entropía Shannon de la
distribución predictiva marginal.

**(c) Mutual Information (BALD)** — separa incertidumbre epistémica de aleatórica
(Houlsby et al. 2011):
$$\text{MI} = H[\bar{p}] - \frac{1}{M}\sum_i H[p_i]$$

In [ ]:
from scipy.stats import norm

def gaussian_class_probs(mu, sigma=2.0, boundaries=OMS_BOUNDS_4):
    """Para cada μ_i, p_i(c) = ∫_{lo_c}^{hi_c} N(y; μ_i, σ²) dy."""
    mu = np.asarray(mu)
    bounds = np.asarray(boundaries, dtype=np.float64)
    z = (bounds[None, :] - mu.reshape(-1, 1)) / sigma          # [N, K+1]
    cdf = norm.cdf(z)
    probs = cdf[:, 1:] - cdf[:, :-1]                            # [N, K]
    probs = probs.reshape(*mu.shape, len(boundaries) - 1)
    probs = np.clip(probs, 1e-12, 1.0)
    return probs / probs.sum(axis=-1, keepdims=True)


def shannon_entropy(p, axis=-1):
    return -np.sum(p * np.log(p), axis=axis)


def compute_metrics_per_image(mc_samples, kernel_width=2.0):
    """Calcula las 3 métricas de confianza por imagen.

    Args:
        mc_samples: [M, N_imgs]
    Returns:
        dict con arrays [N_imgs]:
          - mean:     media MC (predicción puntual final)
          - variance: σ² predictiva
          - entropy:  entropía categórica predictiva (incertidumbre TOTAL)
          - mi:       BALD — incertidumbre EPISTÉMICA
    """
    M, N = mc_samples.shape
    mean = mc_samples.mean(axis=0)
    variance = mc_samples.var(axis=0, ddof=0)

    # Distribución sobre 4 clases OMS por cada muestra
    p_per_sample = gaussian_class_probs(
        mc_samples.reshape(-1), sigma=kernel_width
    ).reshape(M, N, 4)
    p_marginal = p_per_sample.mean(axis=0)                       # [N, 4]
    entropy_total = shannon_entropy(p_marginal, axis=-1)         # [N]
    entropy_per_sample = shannon_entropy(p_per_sample, axis=-1)  # [M, N]
    mi = entropy_total - entropy_per_sample.mean(axis=0)         # [N]

    return {"mean": mean, "variance": variance, "entropy": entropy_total, "mi": mi}


## 6. Score compuesto + validación con risk-coverage curve

Cada métrica se normaliza por **percentil empírico** sobre val (robusto a
outliers). Score = 1 - promedio de los percentiles.

In [ ]:
class ConfidenceScorer:
    METRIC_KEYS = ("variance", "entropy", "mi")

    def __init__(self, sorted_vals):
        self.sorted_vals = sorted_vals

    @classmethod
    def fit(cls, val_metrics):
        return cls({k: np.sort(val_metrics[k]) for k in cls.METRIC_KEYS})

    def _percentile(self, key, x):
        ref = self.sorted_vals[key]
        idx = np.searchsorted(ref, x, side="left")
        return idx.astype(np.float64) / len(ref)

    def score(self, metrics):
        """Devuelve array [N] en [0,1]. 1 = más confianza."""
        perc = np.stack([self._percentile(k, metrics[k]) for k in self.METRIC_KEYS], 0)
        return 1.0 - perc.mean(axis=0)


def risk_coverage_curve(confidence, errors, n_points=100):
    """Ordena por confianza desc., toma top-k%, reporta MAE. Devuelve (cov, mae, aurc)."""
    order = np.argsort(-confidence)
    sorted_err = errors[order]
    cumulative_mae = np.cumsum(np.abs(sorted_err)) / np.arange(1, len(errors) + 1)
    coverages = np.linspace(0.05, 1.0, n_points)
    idx = np.clip((coverages * len(errors)).astype(int) - 1, 0, len(errors) - 1)
    mae = cumulative_mae[idx]
    aurc = float(np.trapz(mae, coverages))
    return coverages, mae, aurc


def calibration_curve(mc, targets, k_values=(0.5, 1.0, 1.5, 2.0, 3.0)):
    """Para cada k, % real de imgs con error <= k*σ predicha vs % teórico."""
    mean = mc.mean(0); std = mc.std(0)
    err  = np.abs(targets - mean)
    return {float(k): (float((err <= k*std).mean()),
                       float(norm.cdf(k) - norm.cdf(-k))) for k in k_values}


### Cargar caché MC y ejecutar el análisis sobre VisualBMI test

El caché contiene las 120 muestras MC ya pre-computadas. Si no existe, lo
generamos (toma ~15 min en GPU).

In [ ]:
vbmi_npz = CACHE_DIR / "mc_samples_visualbmi.npz"

if vbmi_npz.exists():
    print(f"Cargando caché: {vbmi_npz.name}")
    d = np.load(vbmi_npz, allow_pickle=True)
    mc_val,  tg_val  = d["mc_val"],  d["tg_val"]
    mc_test, tg_test = d["mc_test"], d["tg_test"]
else:
    print("Caché no encontrado — generando MC samples (~15 min)...")
    from src.dataset import VisualBMIDataset, get_transforms
    from torch.utils.data import DataLoader
    def loader(df):
        ds = VisualBMIDataset(df=df, transform=get_transforms("val", IMAGE_SIZE), task="regression")
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
    models = load_bayes_ensemble()
    mc_val, tg_val   = mc_predict_ensemble(models, loader(val_df))
    mc_test, tg_test = mc_predict_ensemble(models, loader(test_df))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(vbmi_npz, mc_val=mc_val.astype(np.float32),
                        mc_test=mc_test.astype(np.float32),
                        tg_val=tg_val.astype(np.float32),
                        tg_test=tg_test.astype(np.float32))

# Calcular métricas, ajustar scorer, evaluar
metrics_val  = compute_metrics_per_image(mc_val,  KERNEL_WIDTH)
metrics_test = compute_metrics_per_image(mc_test, KERNEL_WIDTH)

scorer = ConfidenceScorer.fit(metrics_val)
conf_test = scorer.score(metrics_test)

mean_test = metrics_test["mean"]
err_test  = tg_test - mean_test
print(f"MAE global test:                 {np.abs(err_test).mean():.4f}")

cov, mae_curve, aurc = risk_coverage_curve(conf_test, err_test)
for pct in [50, 25, 10]:
    i = int(pct/100 * len(cov)) - 1
    print(f"MAE manteniendo top {pct:3d}% más confiable: {mae_curve[i]:.4f}")
print(f"AURC (menor = mejor): {aurc:.4f}")

cal = calibration_curve(mc_test, tg_test)
print(f"\nCalibración (% observado vs teórico):")
for k, (obs, exp) in cal.items():
    print(f"  μ ± {k}σ:  observado={obs*100:5.1f}%  vs teórico={exp*100:5.1f}%")


### Curva risk-coverage rápida (inline)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cov*100, mae_curve, linewidth=2.5, color="#3498db",
        label=f"VisualBMI test (AURC={aurc:.3f})")
ax.set_xlabel("Cobertura retenida (%)")
ax.set_ylabel("MAE en subconjunto retenido (kg/m²)")
ax.set_title("Risk-coverage curve — el score de confianza es válido si\n"
             "el MAE BAJA al filtrar los menos confiables", fontweight="bold")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 7. MC dropout sobre el clasificador determinista (3 clases)

**Por qué esta sección.** Las métricas de **entropía** y **mutual information**
son matemáticamente más naturales sobre un clasificador (distribución sobre clases
discretas). En este proyecto el clasificador *bayesiano* fracasó (acc 33-36 %,
nivel azar), pero podemos obtener un equivalente bayesiano del clasificador
**determinista** activando dropout en evaluación: **MC dropout** (Gal & Ghahramani
2016 demuestran que esto aproxima la posterior bayesiana sobre los pesos).

Resultado: confianza también para la tarea de clasificación 3-clases, usando el
mejor clasificador del proyecto (`visualbmi_clf3_seed42`, test acc 56.6 %).

In [ ]:
# Construir el clasificador determinista (misma arquitectura que el regresor pero
# con num_classes=3 en la cabeza final)
class BMIClassifier(nn.Module):
    def __init__(self, backbone_name="vit_base_patch16_224.augreg_in21k_ft_in1k",
                 num_classes=3, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0)
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))


def bmi_to_class(bmi: float, n_classes: int = 3) -> int:
    if n_classes == 3:
        if bmi < 25: return 0
        if bmi < 30: return 1
        return 2
    raise ValueError("solo 3 clases en este notebook")


@torch.no_grad()
def mc_dropout_classifier_probs(model, dataloader, n_samples=30, device="cuda"):
    """
    Activa dropout en eval (model.train()) y hace N forwards por imagen.
    Devuelve probabilidades softmax de shape [n_samples, N_imgs, n_classes].
    """
    # Truco: model.train() activa Dropout, pero queremos BatchNorm en modo eval.
    # En este modelo (ViT) no hay BatchNorm; LayerNorm se comporta igual en train/eval.
    model.train()
    all_probs, all_targets, all_bmis = [], [], []
    for images, bmis in dataloader:
        images = images.to(device, non_blocking=True)
        # n_samples forwards → [n_samples, B, C]
        logits_samples = torch.stack(
            [model(images) for _ in range(n_samples)], dim=0
        )
        probs = torch.softmax(logits_samples, dim=-1)
        all_probs.append(probs.cpu().numpy())
        # Convertir BMI a clase discreta (3 clases) para etiquetas verdaderas
        all_targets.extend([bmi_to_class(float(b), 3) for b in bmis])
        all_bmis.extend(bmis.tolist())
    return (np.concatenate(all_probs, axis=1),    # [n_samples, N, 3]
            np.array(all_targets),                 # [N]
            np.array(all_bmis))                    # [N]


def classifier_uncertainty(mc_probs):
    """
    A partir de [n_samples, N, C] computa:
      - mean_probs [N, C]: distribución marginal
      - pred [N]:          clase predicha (argmax del marginal)
      - entropy [N]:       entropía total H[mean_probs]
      - mi [N]:            BALD = H[mean] - mean(H[per_sample])
    """
    mean_probs = mc_probs.mean(axis=0)
    pred = mean_probs.argmax(axis=-1)
    H_mean = shannon_entropy(mean_probs, axis=-1)
    H_per  = shannon_entropy(mc_probs, axis=-1)            # [n_samples, N]
    mi = H_mean - H_per.mean(axis=0)
    return mean_probs, pred, H_mean, mi


### Cargar el clasificador y correr MC dropout sobre test

In [ ]:
clf_ckpt_path = RUNS_DIR / "visualbmi_clf3_seed42" / "best.pt"
clf_cache_path = CACHE_DIR / "mc_dropout_clf3_test.npz"

if not clf_ckpt_path.exists():
    print(f"[!] No existe el checkpoint del clasificador: {clf_ckpt_path}")
    print("    Ese modelo se entrenó antes con src/train_clf.py --num_classes 3")
else:
    if clf_cache_path.exists():
        print(f"Cargando caché MC dropout: {clf_cache_path.name}")
        dc = np.load(clf_cache_path, allow_pickle=True)
        mc_probs_clf, y_true_clf, bmi_clf = dc["mc_probs"], dc["y_true"], dc["bmi"]
    else:
        print("Generando MC dropout (30 forwards × N_test imágenes)...")
        from src.dataset import VisualBMIDataset, get_transforms
        from torch.utils.data import DataLoader
        clf_test_loader = DataLoader(
            VisualBMIDataset(df=test_df, transform=get_transforms("val", IMAGE_SIZE),
                              task="regression"),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
        )
        ckpt = torch.load(clf_ckpt_path, map_location="cpu", weights_only=False)
        clf = BMIClassifier(num_classes=3).to(device)
        clf.load_state_dict(ckpt["model"])
        mc_probs_clf, y_true_clf, bmi_clf = mc_dropout_classifier_probs(
            clf, clf_test_loader, n_samples=N_MC_SAMPLES, device=device
        )
        np.savez_compressed(clf_cache_path,
                            mc_probs=mc_probs_clf.astype(np.float32),
                            y_true=y_true_clf, bmi=bmi_clf)
        print(f"Caché guardado: {clf_cache_path}")

    mean_p, pred, H_clf, MI_clf = classifier_uncertainty(mc_probs_clf)
    acc = float((pred == y_true_clf).mean())
    # confianza = 1 - max prob normalizada por percentil sobre val? Aquí simplificamos:
    # confianza puntual = 1 - entropía normalizada
    H_norm = H_clf / np.log(3)
    conf_clf = 1 - H_norm

    print(f"\nClasificador 3-clases (test):")
    print(f"  Accuracy global:             {acc*100:.1f}%")
    print(f"  Entropía media:              {H_clf.mean():.3f}  (max teórico {np.log(3):.3f})")
    print(f"  Mutual information media:    {MI_clf.mean():.4f}")

    # Risk-coverage para CLASSIFICATION: ¿sube acc al filtrar menos confiables?
    order = np.argsort(-conf_clf)
    correct_sorted = (pred == y_true_clf)[order]
    cum_acc = np.cumsum(correct_sorted) / np.arange(1, len(correct_sorted)+1)
    for pct in [50, 25, 10]:
        i = int(pct/100 * len(cum_acc)) - 1
        print(f"  Accuracy top {pct:3d}% confiable: {cum_acc[i]*100:.1f}%")


### Gráfico de MC dropout sobre el clasificador

Generamos un PNG nuevo que va a `entregable/plots/clf_mc_dropout.png` con
dos paneles: (a) coverage vs accuracy y (b) histograma de entropía.

In [ ]:
if clf_ckpt_path.exists():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("MC dropout sobre visualbmi_clf3_seed42 (clasificador determinista, 3 clases)", fontweight="bold")

    # Panel (a): coverage vs accuracy
    coverages = np.linspace(0.05, 1.0, 50)
    accs = []
    for c in coverages:
        i = int(c * len(cum_acc)) - 1
        accs.append(cum_acc[max(i, 0)])
    ax1.plot(coverages*100, np.array(accs)*100, linewidth=2.5, color="#27ae60")
    ax1.set_xlabel("Cobertura retenida (%)")
    ax1.set_ylabel("Accuracy (%) sobre subconjunto retenido")
    ax1.set_title(f"Coverage-Accuracy  (acc global {acc*100:.1f}%)")
    ax1.axhline(acc*100, color="gray", ls=":", alpha=0.7, label=f"baseline {acc*100:.1f}%")
    ax1.grid(alpha=0.3); ax1.legend()

    # Panel (b): histograma de entropía por clase predicha
    colors = ["#2ecc71", "#f39c12", "#e74c3c"]
    for c in range(3):
        m = pred == c
        ax2.hist(H_clf[m], bins=25, alpha=0.5, color=colors[c],
                 label=f"{OMS_NAMES_3[c]} (n={m.sum()})")
    ax2.set_xlabel("Entropía predictiva  (nats)")
    ax2.set_ylabel("# imágenes")
    ax2.set_title("Entropía por clase predicha")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = PROJECT_ROOT / "entregable" / "plots" / "clf_mc_dropout.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nGráfico guardado: {plot_path}")


## 8. Evaluación cross-dataset (IDOC Mugshots, zero-shot)

69,620 mugshots con altura/peso autodeclarados (booking sheet del Illinois Dept. of
Corrections, ~93 % varones). Evaluamos sin reentrenar — pura generalización fuera
de distribución. El cache de muestras MC ya está pre-computado.

In [ ]:
idoc_cache = CACHE_DIR / "mc_samples_idoc.npz"
if idoc_cache.exists():
    di = np.load(idoc_cache, allow_pickle=True)
    mc_idoc, tg_idoc = di["mc_idoc"], di["tg_idoc"]

    metrics_idoc = compute_metrics_per_image(mc_idoc, KERNEL_WIDTH)
    conf_idoc = scorer.score(metrics_idoc)
    err_idoc  = tg_idoc - metrics_idoc["mean"]

    print(f"IDOC zero-shot (n={len(tg_idoc)}):")
    print(f"  MAE global: {np.abs(err_idoc).mean():.4f}")
    cov_i, mae_i, aurc_i = risk_coverage_curve(conf_idoc, err_idoc)
    for pct in [50, 25, 10]:
        i = int(pct/100 * len(cov_i)) - 1
        print(f"  MAE top {pct:3d}% confiable: {mae_i[i]:.4f}")
    print(f"  AURC: {aurc_i:.4f}")

    print(f"\nComparativa de mediana del score de confianza:")
    print(f"  VisualBMI test: {np.median(conf_test):.3f}")
    print(f"  IDOC zero-shot: {np.median(conf_idoc):.3f}")
    print(f"  → diferencia: {np.median(conf_test)-np.median(conf_idoc):.3f}  "
          f"(positivo = modelo 'sabe que no sabe' en IDOC)")
else:
    print("Caché IDOC no encontrado. Generarlo con:")
    print("  venv/Scripts/python.exe -u src/prepare_faces_idoc.py")
    print("  venv/Scripts/python.exe -u scripts/cache_mc_idoc.py")


## 9. Entrenamientos deterministas (referencia a `src/`)

El regresor determinista (ViT-B y ViT-L con buenas prácticas) y el clasificador 3
clases tienen ~700 líneas cada uno entre `src/train.py` y `src/train_clf.py`.
Copiarlas inline haría el notebook ilegible. Para reentrenar, llamar los scripts
desde el shell o usar las funciones importadas. Aquí dejamos los comandos:

In [ ]:
# Comandos para reentrenar (no se ejecutan desde el notebook por defecto)
TRAIN_COMMANDS = {
    "ViT-B/16 determinista (1 seed, ~75 min)": (
        'venv/Scripts/python.exe -u src/train.py --csv_path visual_bmi_dedup.csv '
        '--backbone vit_base_patch16_224.augreg_in21k_ft_in1k --batch_size 8 '
        '--epochs 60 --freeze_epochs 10 --lr 1e-4 --dropout 0.5 --balance_gender '
        '--ema_decay 0.999 --grad_clip 1.0 --warmup_steps 500 '
        '--seed 42 --run_name visualbmi_vitb_seed42'
    ),
    "ViT-L/16 determinista (1 seed, ~78 min)": (
        'venv/Scripts/python.exe -u src/train.py --csv_path visual_bmi_dedup.csv '
        '--backbone vit_large_patch16_224.augreg_in21k_ft_in1k --batch_size 32 '
        '--epochs 60 --freeze_epochs 5 --lr 5e-5 --dropout 0.3 --balance_gender '
        '--ema_decay 0.999 --grad_clip 1.0 --warmup_steps 200 --slim_ckpt '
        '--seed 42 --run_name visualbmi_vitl_seed42'
    ),
    "Clasificador 3-clases (~75 min)": (
        'venv/Scripts/python.exe -u src/train_clf.py --csv_path visual_bmi_dedup.csv '
        '--num_classes 3 --epochs 60 --batch_size 8 --seed 42 '
        '--run_name visualbmi_clf3_seed42'
    ),
}
for desc, cmd in TRAIN_COMMANDS.items():
    print(f"### {desc}\n{cmd}\n")


## 10. Generación de los 9 + 1 plots PNG en inglés

Reproduce todos los PNGs del entregable. Si se modificó algo arriba, basta con
re-ejecutar las dos celdas siguientes para regenerar todo.

In [ ]:
import subprocess
# 4 plots base (dataset, splits, confusion matrices, model comparison)
subprocess.run(["venv/Scripts/python.exe", "-u", "src/plot_data_en.py"],
               cwd=str(PROJECT_ROOT), check=True)
# 5 plots de incertidumbre del regresor bayesiano
subprocess.run(["venv/Scripts/python.exe", "-u", "src/plot_uncertainty.py"],
               cwd=str(PROJECT_ROOT), check=True)
# El gráfico de MC dropout sobre clasificador (sección 7) ya se guardó arriba.
print("\nTodos los plots en entregable/plots/")


---

**Notebook completo.** Para experimentar:

1. Cambiar parámetros editables en la sección 0.
2. `TRAIN_BAYES_REG = True` para reentrenar el regresor bayesiano (cualquier
   cambio en la arquitectura BayesianHead se refleja al reentrenar).
3. Modificar `KERNEL_WIDTH` para cambiar el ancho del kernel gaussiano en la
   marginalización OMS.
4. Cambiar la lista `SEEDS` para usar más o menos modelos en el ensemble.

**Referencias:**

- Houlsby et al. 2011 — BALD (mutual information).
- Gal & Ghahramani 2016 — MC dropout como aproximación bayesiana.
- Lakshminarayanan et al. 2017 — Deep ensembles.
- Geifman & El-Yaniv 2017 — Selective prediction / risk-coverage.
